In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset , DataLoader
from  torch import nn

In [ ]:
from google.colab import files
file_uploaded = files.upload()

Saving 100_Unique_QA_Dataset.csv to 100_Unique_QA_Dataset (1).csv


In [ ]:
df = pd.read_csv("/content/100_Unique_QA_Dataset.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [ ]:
def tokenize(text):
  text = text.lower()
  text= text.replace("?","")
  text = text.replace("'","")
  return text.split()

In [ ]:
tokenize('Which country is known for the Eiffel Tower?')

['which', 'country', 'is', 'known', 'for', 'the', 'eiffel', 'tower']

In [ ]:
# vocab
vocab = {"<UNK>":0}

In [ ]:
def build_vocab(row):
  token_question = tokenize(row['question'])
  token_answer = tokenize(row['answer'])
  merge_token = token_question + token_answer
  for token in merge_token:
    if token not in vocab:
      vocab[token] = len(vocab)

In [ ]:
df.apply(build_vocab, axis= 1 )

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [ ]:
vocab

{'<UNK': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unite

In [ ]:
def text_to_index(text , vocab):
  index_text = []
  for token in tokenize(text):
    if token in vocab:
      index_text.append(vocab[token])
    else:
      index_text.append(vocab["<UNK>"])
  return index_text


In [ ]:
text_to_index("what is hunain ", {"<UNK>": 0, **{k:v for k,v in vocab.items() if k != '<UNK>'}})

[1, 2, 0]

In [ ]:
class QAdataset(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab
  def __len__(self):
    return len(self.df)
  def __getitem__(self, idx):
    question =text_to_index(self.df.iloc[idx]['question'], self.vocab)
    answer = text_to_index(self.df.iloc[idx]['answer'], self.vocab)
    return torch.tensor(question) , torch.tensor(answer)

In [ ]:
dataset = QAdataset(df , vocab)

In [ ]:
dataload = DataLoader(dataset , batch_size=1 , shuffle=True)

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self , vocab):
        super().__init__()
        self.embeed = nn.Embedding(vocab , 50)
        self.rnn = nn.RNN(50 , 64 , batch_first=True )
        self.lr = nn.Linear(64 , vocab)
    def forward(self , question):
        embeed = self.embeed(question)
        hidden , final = self.rnn(embeed)
        return self.lr(final.squeeze(0))

In [ ]:
model = SimpleRNN(len(vocab))
loss_fn  = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters() , lr=0.001)

In [ ]:
for epoch in range(20):
  train_loss = 0
  model.train()
  for qu , ans in dataload:
    y_pred = model(qu)
    loss = loss_fn(y_pred , ans[0])
    train_loss += loss.item()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  print(f"epoch {epoch} loss {train_loss/len(dataload)}")



epoch 0 loss 5.784253719117906
epoch 1 loss 5.062197433577643
epoch 2 loss 4.195570508639018
epoch 3 loss 3.455997019343906
epoch 4 loss 2.852490516503652
epoch 5 loss 2.3119176652696396
epoch 6 loss 1.8233914070659214
epoch 7 loss 1.4122697989145914
epoch 8 loss 1.073520384894477
epoch 9 loss 0.8219593971967697
epoch 10 loss 0.634331308470832
epoch 11 loss 0.4962347144881884
epoch 12 loss 0.3989741929703289
epoch 13 loss 0.323649800899956
epoch 14 loss 0.2672439375685321
epoch 15 loss 0.22787240975432926
epoch 16 loss 0.1928948011663225
epoch 17 loss 0.1653491201914019
epoch 18 loss 0.140814676342739
epoch 19 loss 0.12442317369083564


In [ ]:
def predict(model , question , threshold=0.5):
  numerical_question = text_to_index(question , vocab)
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)
  output = model(question_tensor)
  probabilities = torch.softmax(output , dim=1)
  val , index = torch.max(probabilities , dim=1)
  if val < threshold:
    return "I don't know"
  else:
    print(list(vocab.keys())[index])


In [ ]:
predict(model," Who discovered gravity?")

newton
